In [0]:
df = spark.read.table("ecommerce_analytics.bronze.sales_orders")
df.display()



**->Array of Array**
```
[
    ["AVpgIu4Q1cnluZ0-xBK-","13"],
    ["AVpfeG5oilAPnD_xcTsG","27"],
    ["AVqVGaEBv8e3D1O-ldFu","64"],
    ["AVpg-Wj61cnluZ0-8sZe","87"],
    ["AVphTO5W1cnluZ0-Aygg","52"],
    ["AVpfMVD-ilAPnD_xW6bu","49"]
]
```

**->Array of Struct or column**
```
[
    {"curr":"USD","id":"AVpfIODe1cnluZ0-eg35","name":"Cyber-shot DSC-WX220 Digital Camera (Black)","price":"218","promotion_info":null,"qty":"3","unit":"pcs"},
    {"curr":"USD","id":"AVpjedgc1cnluZ0-W4NI","name":"Rony MEXM100BT 160W RMS Marine CD Receiver with Bluetooth (Black) and SiriusXM Ready","price":"293","promotion_info":null,"qty":"1","unit":"pcs"},
    {"curr":"USD","id":"AVpfdBS41cnluZ0-lBIj","name":"Details About Mogitech G920 Xbox Driving Force Racing Wheel For Xbox One And Pc (941000121)","price":"239","promotion_info":null,"qty":"2","unit":"pcs
]
```
    

**->Array of Struct or column**
```
[
    {"promo_disc":0.03,"promo_id":"0","promo_item":"AVpfMVD-ilAPnD_xW6bu","promo_qty":"2"}
]
```

### three important functions

* explode = convert one row to multiple rows
* from_json = to convert string to json or string tpo structured column
* explode_outer = same as ewxplode but also handles if data is null (better to use this)

### Ordered Products

```
[
    {"curr":"USD","id":"AVpfuJ4pilAPnD_xhDyM","name":"Rony LBT-GPX555 Mini-System with Bluetooth and NFC","price":"993","promotion_info":null,"qty":"3","unit":"pcs"},
 
    {"curr":"USD","id":"AVpe6jFBilAPnD_xQxO2","name":"Aeon 71.5 x 130.9 16:9 Fixed Frame Projection Screen with CineWhite Projection Surface","price":"218","promotion_info":null,"qty":"3","unit":"pcs"},
 
    {"curr":"USD","id":"AVpfIODe1cnluZ0-eg35","name":"Cyber-shot DSC-WX220 Digital Camera (Black)","price":"448","promotion_info":null,"qty":"2","unit":"pcs"}
]
```

**-> We will use to define schema using = StructType, StructField, ArrayType**

In [0]:
df.select("ordered_products").display()

In [0]:


# from pyspark.sql.types import * 
from pyspark.sql.types import StructType, StructField, StringType, ArrayType
from pyspark.sql.functions import from_json, col

ordered_products_schema = ArrayType(
    StructType([
        StructField("curr", StringType()),
        StructField("id", StringType()),
        StructField("name", StringType()),
        StructField("price", StringType()),
        StructField("promotion_info", StringType()),
        StructField("qty", StringType()),
        StructField("unit", StringType())
    ])
)
#  Attach with column by creating new ones

df_parsed = df.withColumn("ordered_products", from_json(col("ordered_products"), ordered_products_schema))

In [0]:

from pyspark.sql.functions import explode_outer
df_exploded_op = df_parsed.withColumn("ordered_products", explode_outer("ordered_products"))
df_exploded_op.display()


In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import from_json, col, explode_outer

# Read table
df = spark.read.table("ecommerce_analytics.bronze.sales_orders")

# Schema for ordered_products
ordered_products_schema = ArrayType(
    StructType([
        StructField("curr", StringType()),
        StructField("id", StringType()),
        StructField("name", StringType()),
        StructField("price", StringType()),
        StructField("promotion_info", StringType()),
        StructField("qty", StringType()),
        StructField("unit", StringType())
    ])
)

# Convert string column into array<struct>
df_parsed = df.withColumn(
    "ordered_products",
    from_json(col("ordered_products"), ordered_products_schema)
)

# Explode array
df_exploded_op = df_parsed.withColumn(
    "ordered_products",
    explode_outer("ordered_products")
)



# Select final columns
df_ordered_products = df_exploded_op.select(
    "customer_id",
    "customer_name",
    "order_number",
    "ordered_products.id",
    col("ordered_products.name").alias("product_name"),
    "ordered_products.price",
    col("ordered_products.curr"),
    col("ordered_products.qty"),
    col("ordered_products.unit")
)

# Display output
display(df_ordered_products)

In [0]:
display(df.select("promo_info"))

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import from_json, col, explode_outer

promo_info_schema = ArrayType(
    StructType([
        StructField("promo_disc", StringType()),
        StructField("promo_id", StringType()),
        StructField("promo_item", StringType()),
        StructField("promo_qty", StringType())

    ])
)

# Parse promo info col

df_promo_parsed = df.withColumn("promo_info", from_json(col("promo_info"), promo_info_schema))

#Exploded promo info array
df_promo_parsed = df_promo_parsed.withColumn("promo_info", explode_outer("promo_info")).filter(col("promo_info").isNotNull())

#select final columns

df_promo_info = df_promo_parsed.select(
    "customer_id",
    "customer_name",
    "order_number",
    col("promo_info.promo_disc").alias("promo_discount"),
    col("promo_info.promo_id").alias("promo_id"),
    col("promo_info.promo_item").alias("promo_item"),
    col("promo_info.promo_qty").alias("promo_quantity")

) 

df_promo_info.display()

### Clicked items

In [0]:
df = spark.read.table("ecommerce_analytics.bronze.sales_orders")
df.select("clicked_items")

In [0]:
from pyspark.sql.types import *
clicked_items_schema = ArrayType(
    ArrayType(StringType())

)

In [0]:
from pyspark.sql.functions import from_json, col, explode_outer
df_parsed_clicked_items = df.withColumn("clicked_items", from_json(col("clicked_items"), clicked_items_schema))

df_parsed_clicked_items.display()

In [0]:
df_exploded_ci = df_parsed_clicked_items.withColumn("clicked_items", explode_outer("clicked_items")).filter(col("clicked_items").isNotNull())

df_exploded_ci.display()


In [0]:
clicked_items = df_exploded_ci.select(
    "customer_id",
    "customer_name",
    "order_number",
    col("clicked_items")[0].alias("product_id"),
    col("clicked_items")[1].alias("score")
    
)

clicked_items.display()